# Moving objects on the FWD Center Lab board

`FWDCenterLabSiva.yaml` inherits the board, rig rails, 3x3 grid and calibration
from `FWDCenterLabMCC.yaml` and repaints its four manipulables to match the ones
actually on the lab table.  This notebook moves those four around the grid while
the simulation runs.

**Why it is "move", not "add".**  MuJoCo freezes `nbody`/`njnt`/`ngeom` at
compile time — there is no `add_body()`, which is why `server.py` still refuses a
runtime `scene_load`.  Every object that will ever exist is declared in the
scene.  What placement does is put a declared object on a named cell, on demand,
with no restart.

Start the sim on this scene first:

```
REACHY_SIM_SCENE=FWDCenterLabSiva REACHY_SIM_DISTORTION=1 ./scripts/start_sim.sh
```

Watch it at **RViz** http://localhost:6080 or **camera** http://localhost:8080.

In [1]:
import asyncio, json, sys, pathlib
import websockets

REPO = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO / "native_mujoco"))
sys.path.insert(0, str(REPO / "src"))

from protocol import Hello, PlaceObject

URI = "ws://127.0.0.1:8765"
SCENE = REPO / "scenes" / "FWDCenterLabSiva.yaml"

## 1. What the scene gives you

Read from the same YAML the simulator loaded, so the notebook and the physics
cannot disagree about where a cell is.

`SceneModel.from_yaml` follows `extends:`, so this sees the parent's board and
grid — not just the four objects this child restates.  It did not always: until
2026-09-09 it used a raw YAML load, and a child scene arrived with no table, no
cells and no rails.

In [2]:
from reachy_ai.scene.awareness import SceneModel
from placement import cells_from_scene, shoulder_distance
from scene_io import load_scene

scene = SceneModel.from_yaml(str(SCENE))
doc = load_scene(str(SCENE))
cells = cells_from_scene(doc)

print(f"table surface z : {scene.table_surface_z:.4f} m")
print(f"rig rails       : {len([o for o in scene.static_obstacles() if 'rig-frame' in o.tags])}")
print()
print("cells (row 1 = nearest the robot, col 1 = its LEFT / +y):")
for name in sorted(cells):
    c = cells[name]
    flag = "" if c.reachable else "   <- right arm cannot reach"
    print(f"   {name}  x={c.x:+.4f}  y={c.y:+.4f}  {c.shoulder_distance_m:.3f} m{flag}")

table surface z : 0.7400 m
rig rails       : 5

cells (row 1 = nearest the robot, col 1 = its LEFT / +y):
   r1c1  x=+0.2794  y=+0.1524  0.489 m
   r1c2  x=+0.2794  y=+0.0000  0.397 m
   r1c3  x=+0.2794  y=-0.1524  0.351 m
   r2c1  x=+0.4318  y=+0.1524  0.589 m
   r2c2  x=+0.4318  y=+0.0000  0.516 m
   r2c3  x=+0.4318  y=-0.1524  0.481 m
   r3c1  x=+0.5842  y=+0.1524  0.709 m   <- right arm cannot reach
   r3c2  x=+0.5842  y=+0.0000  0.649 m   <- right arm cannot reach
   r3c3  x=+0.5842  y=-0.1524  0.622 m


In [3]:
print("manipulables, and the cell each one starts on:")
for obj in doc["objects"]:
    if "manipulable" not in (obj.get("tags") or []):
        continue
    x, y, z = obj["pose"]["position"]
    on = [n for n, c in cells.items()
          if abs(x - c.x) <= c.half_extent and abs(y - c.y) <= c.half_extent]
    print(f"   {obj['id']:14s} {obj['semantic_class']:9s} -> {on[0] if on else 'off-grid'}")

occupied = {n for n in cells
            for o in doc["objects"]
            if "manipulable" in (o.get("tags") or [])
            and abs(o["pose"]["position"][0] - cells[n].x) <= cells[n].half_extent
            and abs(o["pose"]["position"][1] - cells[n].y) <= cells[n].half_extent}
reachable = {n for n in cells if cells[n].reachable}

print("\nreachable :", sorted(reachable))
print("occupied  :", sorted(occupied))
print("free now  :", sorted(reachable - occupied))

manipulables, and the cell each one starts on:
   red_cube       cube      -> r1c3
   blue_cylinder  cylinder  -> r2c2
   soda_can       can       -> r2c1
   foam_block     foam      -> r1c2

reachable : ['r1c1', 'r1c2', 'r1c3', 'r2c1', 'r2c2', 'r2c3', 'r3c3']
occupied  : ['r1c2', 'r1c3', 'r2c1', 'r2c2']
free now  : ['r1c1', 'r2c3', 'r3c3']


## 2. A client

`place` waits for the ack rather than firing and hoping.  The ack carries
`sim_step` — the step the object actually landed on — so a later camera frame can
be lined up against a placement whose position you already know.  That is what
makes this a perception check rather than a demo.

Acks are routed back to the connection that asked.  Worth knowing because they
were not, briefly: a single shared queue meant the Docker bridge could swallow a
notebook's ack, and the notebook would wait forever for a placement that had
already happened.

In [4]:
class Board:
    def __init__(self, ws):
        self.ws = ws

    async def _ack(self, rid):
        while True:
            msg = json.loads(await self.ws.recv())
            if msg["type"] == "place_ack" and msg["request_id"] == rid:
                return msg

    async def place(self, object_id, cell, yaw_deg=0.0, **kw):
        rid = f"{object_id}@{cell}"
        await self.ws.send(PlaceObject(object_id=object_id, cell=cell,
                                       yaw_deg=yaw_deg, request_id=rid,
                                       **kw).encode())
        ack = await self._ack(rid)
        if not ack["accepted"]:
            raise RuntimeError(ack["error"])
        return ack

    async def home(self, object_id):
        """Back to where the scene declared it."""
        rid = f"{object_id}@home"
        await self.ws.send(PlaceObject(object_id=object_id, cell=None,
                                       request_id=rid).encode())
        return await self._ack(rid)

    async def poses(self):
        while True:
            msg = json.loads(await self.ws.recv())
            if msg["type"] == "state":
                return {o["object_id"]: o["pos_xyz"] for o in msg["objects"]}


async def connected(fn):
    async with websockets.connect(URI, max_size=None) as ws:
        await ws.send(Hello().encode()); await ws.recv()
        return await fn(Board(ws))

## 3. Move one object

`red_cube` starts on `r1c3`.  `r1c1` is free and reachable, so move it there.

In [5]:
async def _move(board):
    ack = await board.place("red_cube", "r1c1")
    p = ack["placement"]
    print(f"red_cube -> {p['cell']}  pos={[round(v, 4) for v in p['position']]}  "
          f"step={ack['sim_step']}")
    return ack

await connected(_move)

red_cube -> r1c1  pos=[0.2794, 0.1524, 0.772]  step=16600


{'type': 'place_ack',
 'request_id': 'red_cube@r1c1',
 'accepted': True,
 'placement': {'object_id': 'red_cube',
  'cell': 'r1c1',
  'position': [0.2794, 0.1524, 0.772],
  'quat_wxyz': [1.0, 0.0, 0.0, 0.0],
  'yaw_deg': 0.0,
  'stowed': False},
 'sim_step': 16600,
 'error': ''}

## 4. What it refuses

Three refusals, each for a different reason.  All of them come back as an ack
with a message, not as a dropped request — a placement that silently does
nothing is the failure mode this is built to avoid.

In [6]:
async def _refusals(board):
    for oid, cell, why in [
        ("blue_cylinder", "r3c1", "unreachable — 71 cm, past the arm's measured reach"),
        ("foam_block",    "r2c2", "occupied — blue_cylinder is already there"),
        ("no_such_thing", "r1c1", "unknown object"),
    ]:
        try:
            await board.place(oid, cell)
            print(f"{oid:14s} {cell}  PLACED")
        except RuntimeError as exc:
            print(f"{oid:14s} {cell}  refused: {str(exc)[:76]}")
            print(f"{'':14s}      ({why})")

await connected(_refusals)

blue_cylinder  r3c1  refused: cell r3c1 is 71 cm from the right shoulder and was measured UNREACHABLE by I
                    (unreachable — 71 cm, past the arm's measured reach)
foam_block     r2c2  refused: cell r2c2 already holds 'blue_cylinder'; stow it first, or pass allow_occupi
                    (occupied — blue_cylinder is already there)


no_such_thing  r1c1  refused: 'no_such_thing' is not a placeable object; the scene's free-joint objects ar
                    (unknown object)


### Overriding a refusal

Both guards take an override, because both refusals are sometimes the point —
testing that a planner *declines* an unreachable target, or deliberately staging
a collision.

In [7]:
async def _override(board):
    ack = await board.place("blue_cylinder", "r3c1", allow_unreachable=True)
    print("placed on r3c1 deliberately:", ack["placement"]["cell"])
    print("the arm still cannot reach it — that is what makes it a useful test")
    await board.home("blue_cylinder")
    print("returned home")

await connected(_override)

placed on r3c1 deliberately: r3c1
the arm still cannot reach it — that is what makes it a useful test


returned home


## 5. Rearrange the board

Cells are refused when occupied, so an object has to leave before another
arrives.  Sending every object home first is the simplest way to get a known
starting state.

In [8]:
LAYOUT = [("red_cube",      "r1c1", 0),
          ("blue_cylinder", "r2c3", 0),
          ("soda_can",      "r1c3", 0),
          ("foam_block",    "r2c2", 20)]

async def _rearrange(board):
    for oid, _, _ in LAYOUT:
        await board.home(oid)
    for oid, cell, yaw in LAYOUT:
        ack = await board.place(oid, cell, yaw_deg=yaw)
        print(f"  {oid:14s} -> {ack['placement']['cell']}  yaw={yaw}")

    poses = await board.poses()
    print()
    for oid, cell, _ in LAYOUT:
        x, y, z = poses[oid]
        c = cells[cell]
        ok = abs(x - c.x) <= c.half_extent and abs(y - c.y) <= c.half_extent
        print(f"  {oid:14s} ({x:+.4f}, {y:+.4f}, {z:.4f})  on {cell}: {ok}")

await connected(_rearrange)

  red_cube       -> r1c1  yaw=0
  blue_cylinder  -> r2c3  yaw=0
  soda_can       -> r1c3  yaw=0


  foam_block     -> r2c2  yaw=20

  red_cube       (+0.2794, +0.1524, 0.7700)  on r1c1: True
  blue_cylinder  (+0.4318, -0.1524, 0.7900)  on r2c3: True
  soda_can       (+0.2794, -0.1524, 0.7974)  on r1c3: True
  foam_block     (+0.4318, -0.0000, 0.7645)  on r2c2: True


## 6. Back to the scene's own layout

In [9]:
async def _reset(board):
    for oid in ("red_cube", "blue_cylinder", "soda_can", "foam_block"):
        await board.home(oid)
    print("board restored to the layout FWDCenterLabSiva declares")

await connected(_reset)

board restored to the layout FWDCenterLabSiva declares


## Ground still to cover

- **Placement is not a grasp.** Reaching a cell and grasping at it are different
  constraints — MCC's sweep is orientation-unconstrained, and the tuned demo
  workspace was only y in [-0.22, +0.02].  Every col-1 cell is suspect for
  grasping even though the tip reaches it.
- **Object sizes are ~1.5x too large** (issue #42), which affects gripper
  aperture and approach clearance, not just how big things look.
- **`cell_r3c3` reachability is unresolved** (issue #41): 0.622 m, labelled
  reachable against a stated 0.609 m maximum.